mask只能在q和k相乘之后做。
如果mask在q和k相乘之前做，反而不一定能得到负无穷小这个结果。因为k是负无穷时，q同样可以是一个负数，二者结果就是正无穷了。

In [12]:
import torch
from torch import nn

from attention_layers import (
    DotProductAttention,
    MultiHeadAttention,
    build_padding_mask,
    build_causal_mask,
)

In [14]:
class MultiHeadAttention2(nn.Module):
    """Multi-head scaled dot-product attention.

    ``hidden_size`` is the feature size of each head and of the final output.
    """

    def __init__(
        self,
        query_size: int,
        key_size: int,
        value_size: int,
        hidden_size: int,
        head_count: int,
    ) -> None:
        super().__init__()
        if head_count <= 0:
            raise ValueError("head_count must be greater than 0")
        if hidden_size % head_count != 0:
            raise ValueError(
                f"hidden_size ({hidden_size}) must be divisible by head_count ({head_count})"
            )
        self.d_head = hidden_size // head_count
        self.wq = nn.Linear(query_size, hidden_size)
        self.wk = nn.Linear(key_size, hidden_size)
        self.wv = nn.Linear(value_size, hidden_size)
        self.wo = nn.Linear(hidden_size, hidden_size)
        self.head_count = head_count
        self.attention = DotProductAttention()
        self.attention_weights: torch.Tensor | None = None

    def forward(
        self,
        queries: torch.Tensor,
        keys: torch.Tensor,
        values: torch.Tensor,
        valid_lens: torch.Tensor | None = None,
        causal: bool = False
    ) -> torch.Tensor:
        projected_queries = self.split_heads_for_attention(self.wq(queries))
        projected_keys = self.split_heads_for_attention(self.wk(keys))
        projected_values = self.split_heads_for_attention(self.wv(values))

        output = self.attention(
            projected_queries,
            projected_keys,
            projected_values,
            valid_lens,
            causal
        )
        batch_size = queries.shape[0]
        self.attention_weights = self.attention.attention_weights.reshape(
            batch_size,
            self.head_count,
            queries.shape[1],
            keys.shape[1],
        )
        return self.wo(self.combine_attention(output))

    def split_heads_for_attention(self, tensor: torch.Tensor) -> torch.Tensor:
        tensor = tensor.reshape(
            tensor.shape[0], tensor.shape[1], self.head_count, self.d_head
        )
        tensor = tensor.permute(0, 2, 1, 3)
        print("attention,4d",tensor)
        tensor = tensor.reshape(-1, tensor.shape[2], tensor.shape[3])
        print("attention,3d",tensor)
        return tensor


    def combine_attention(self, tensor: torch.Tensor) -> torch.Tensor:
        tensor = tensor.reshape(
            tensor.shape[0] // self.head_count,
            self.head_count,
            tensor.shape[1],
            tensor.shape[2],
        )
        tensor = tensor.permute(0, 2, 1, 3)
        return tensor.reshape(tensor.shape[0], tensor.shape[1], -1)

In [ ]:
import torch 

torch.manual_seed(0)

batch, seq_q, seq_k = 2, 4, 4
query_size, key_size, value_size = 8, 8, 8
hidden_size, head_count = 16, 4

mha = MultiHeadAttention2(query_size, key_size, value_size, hidden_size, head_count)
mha.eval()

q = torch.randn(batch, seq_q, query_size)
print("q.shape", q.shape)
print("q[0]", q[0])
k = torch.randn(batch, seq_k, key_size)
v = torch.randn(batch, seq_k, value_size)
pq = mha.wq(q)
print("wq(q).shape", pq.shape)
print("wq(q)", pq)
pq_heads = mha.split_heads_for_attention(pq)   # (batch*head, seq_q, d_head)
print("split_heads 后:", pq_heads.shape, " d_head =", mha.d_head)

pk_heads = mha.split_heads_for_attention(mha.wk(k))
pv_heads = mha.split_heads_for_attention(mha.wv(v))

scores = torch.bmm(pq_heads, pk_heads.transpose(1, 2)) / mha.d_head**0.5
print("scores:", scores.shape)  # (batch*head, seq_q, seq_k)




q.shape torch.Size([2, 4, 8])
q[0] tensor([[ 0.5842,  1.0504,  1.2856, -1.6165, -0.7690, -1.2205,  0.5731,  0.6992],
        [ 0.2511,  0.2785, -0.0946,  1.6104, -0.1217, -1.3941, -0.9048, -0.3467],
        [ 0.7049,  0.0305, -0.8542,  0.5388, -0.5265, -1.3320,  1.5451,  0.4086],
        [-2.0546,  0.5259,  0.5995, -0.4078,  0.4530, -0.3918,  2.1403, -0.2062]])
wq(q).shape torch.Size([2, 4, 16])
attention,4d tensor([[[[ 0.3015,  0.2363, -0.1438,  1.4383],
          [-0.6695,  0.1214, -0.6571, -0.3162],
          [ 0.0378,  0.0147, -0.3114, -0.2654],
          [-0.2585, -0.5877, -0.5989,  0.1143]],

         [[-0.3194,  1.1045, -0.3053, -0.6475],
          [-1.4438, -0.5897,  0.5288, -0.4323],
          [-0.3466,  0.5671,  0.2821, -0.4409],
          [ 0.2317,  0.4092,  1.1559, -0.8418]],

         [[-0.8685,  0.4347,  1.0001, -0.7550],
          [-0.0318,  1.0265, -0.3334,  0.6573],
          [-0.8637,  0.3140, -0.6823, -0.6439],
          [-1.0278,  0.1628,  1.1809, -0.7987]],

      

In [6]:
out = mha(q, k, v, causal=True)
print("最终输出:", out.shape)                        # (batch, seq_q, hidden)
print("attention_weights:", mha.attention_weights.shape)  # (batch, head, seq_q, seq_k)

# 看第 0 个样本、第 0 个头的注意力矩阵，应该是下三角（上三角被 mask 成 0）
w = mha.attention_weights[0, 0]
print(w.round(decimals=3))
print("每行是否和为 1:", w.sum(dim=-1))

最终输出: torch.Size([2, 4, 16])
attention_weights: torch.Size([2, 4, 4, 4])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.4880, 0.5120, 0.0000, 0.0000],
        [0.3180, 0.3430, 0.3380, 0.0000],
        [0.2460, 0.1900, 0.2380, 0.3260]], grad_fn=<RoundBackward1>)
每行是否和为 1: tensor([1.0000, 1.0000, 1.0000, 1.0000], grad_fn=<SumBackward1>)


In [7]:
valid_lens = torch.tensor([3, 2])  # 样本0有效长度3，样本1有效长度2

# MultiHeadAttention2：多头下会因为 batch 维对不上而报错
try:
    mha(q, k, v, valid_lens=valid_lens)
except RuntimeError as e:
    print("MultiHeadAttention2 padding 报错(符合预期):", e)

# 库里的 MultiHeadAttention：做了 repeat_interleave，能正常跑
from attention_layers import MultiHeadAttention
mha_ok = MultiHeadAttention(query_size, key_size, value_size, hidden_size, head_count)
mha_ok.eval()
out_ok = mha_ok(q, k, v, valid_lens=valid_lens)
print("MultiHeadAttention padding 输出:", out_ok.shape)
print("样本1 第0头权重(后2列应为0):")
print(mha_ok.attention_weights[1, 0].round(decimals=3))

MultiHeadAttention2 padding 报错(符合预期): The size of tensor a (2) must match the size of tensor b (8) at non-singleton dimension 0
MultiHeadAttention padding 输出: torch.Size([2, 4, 16])
样本1 第0头权重(后2列应为0):
tensor([[0.4190, 0.5810, 0.0000, 0.0000],
        [0.5950, 0.4050, 0.0000, 0.0000],
        [0.3540, 0.6460, 0.0000, 0.0000],
        [0.5160, 0.4840, 0.0000, 0.0000]], grad_fn=<RoundBackward1>)
